# Data Preprocess 

This script concatenates most recent passive data with backup data and performs basic preprocessing on passive, EMA and Monitoring data.

In [ ]:
from pyprojroot import here # define relative paths to the project root (working directory)
from pathlib import Path
import sys
from datetime import date
import pandas as pd
import gc  
import os
import glob
import numpy as np
import pickle

# --- Paths / imports -------------------------------------------------

# relative project root
PROJECT_ROOT = here() # '.here' is located as invisible file in the project root working directory
PREPROCESSING_DIR = PROJECT_ROOT / "functions" / "preprocessing"
for p in (PROJECT_ROOT, PREPROCESSING_DIR):
    if str(p) not in sys.path:
        sys.path.append(str(p))

from server_config import datapath, proj_sheet, preprocessed_path, raw_path, backup_path

from functions.preprocessing.infer_timeoffset import (
    create_utcday_tzoffset_df,
    merge_fill_tz,
)

# --- Dates ------------------------------------------------------------
today_str = date.today().strftime("%d%m%Y")        
today_day = pd.Timestamp.today().normalize()       


# --- Path -------------------------------------------------------------

datapath = Path(raw_path) / f"export_tiki_{today_str}"  

ImportError: cannot import name 'preprocessed_path_freezed' from 'server_config' (/home/leha18/tiki_code/server_config.py)

## 1. Passive Data

### 1.1 Load most recent passive data

In [ ]:
# actual passive + ema_data

# define the pattern for passive data files
file_pattern = os.path.join(datapath, "epoch_part*.csv")

# use glob to find all matching files
file_list = glob.glob(file_pattern)

# sort the file list for consistent ordering
file_list.sort()

# concatenate all CSV files into a single DataFrame
df_complete = pd.concat((pd.read_csv(f, encoding="latin-1", low_memory=False) for f in file_list), ignore_index=True)


In [ ]:
# Extract customer identifier and reduce to first 4 characters
df_complete["customer"] = df_complete.customer.str.split("@").str.get(0)
df_complete["customer"] = df_complete["customer"].str[:4]

for col in ["startTimestamp", "endTimestamp"]:
    df_complete[col] = (
        pd.to_datetime(df_complete[col], utc=True, errors="coerce", unit="ms")
    )

#df_complete

In [ ]:
df_complete = df_complete[['customer', 'startTimestamp', 'endTimestamp', 'timezoneOffset', 'type',
       'stringValue', 'booleanValue', 'doubleValue', 'longValue']]

### 1.2 Load big backup dataset

In [ ]:
# merge with backup data
backup_path = preprocessed_path + "/backup_passive_05052025.parquet"

# backup_path = preprocessed_path + "/backup_passive_last.feather"
df_backup = pd.read_parquet(backup_path)

In [ ]:
# make independent copies of both DataFrames to avoid SettingWithCopyWarning (future modifications do not affect any original DataFrame)
df_backup = df_backup.copy()
df_complete = df_complete.copy()

# convert booleanValue to boolean[pyarrow] dtype before concatenation so that it can be saved to .feather later
# alternative to "boolean[pyarrow]" is "boolean", but it is experimental and may change in future pandas versions
df_backup['booleanValue'] = df_backup['booleanValue'].astype('boolean[pyarrow]')
df_complete['booleanValue'] = df_complete['booleanValue'].astype('boolean[pyarrow]')

In [ ]:
# latest timestamp from the backup dataset
latest_timestamp = df_backup['startTimestamp'].max()

# filter from df_complete only those entries that are newer than what’s already in the backup
df_complete_filtered = df_complete[df_complete['startTimestamp'] > latest_timestamp]

### 1.3 Concat Backup and most recent data

In [ ]:
# update the backup by concatenating only the newly filtered rows from df_complete, creating an up-to-date backup
df_backup_recent = pd.concat([df_backup, df_complete_filtered], ignore_index=True)

### 1.4 Rename variable names and create additional columns 

In [ ]:
# define a clear mapping for rename columns
rename_columns = {"customer": "id",
                  "type": "modality",
                  "startTimestamp": "timestamp_start",
                  "endTimestamp": "timestamp_end",
                  "booleanValue": "boolean_value",
                  "doubleValue": "double_value",
                  "longValue": "long_value",
                  "timezoneOffset": "timezone_offset"}

# apply renaming
df_backup_recent = df_backup_recent.rename(columns=rename_columns)

In [ ]:
# create a unified float_value column:
# use 'doubleValue' where available (more precise), otherwise use 'longValue'
df_backup_recent['float_value'] = df_backup_recent['double_value'].fillna(df_backup_recent['long_value'])


In [ ]:
# drop original value columns that have been unified into 'float_value' + 'stringValue' (because only ECG data are stored as string for period March - November 2023) + 'createdtAt'
df_backup_recent = df_backup_recent.drop(columns=['double_value', 'long_value', 'stringValue'])


In [ ]:
# create a time_interval (duration in seconds) column
df_backup_recent['time_interval'] = (
    df_backup_recent['timestamp_end'] - df_backup_recent['timestamp_start']
).dt.total_seconds()

# create a start_date and start_hour column
df_backup_recent['start_date']  = df_backup_recent['timestamp_start'].dt.normalize()
df_backup_recent['start_hour'] = df_backup_recent['timestamp_start'].dt.hour

What's next?

1. Since we want to include 'for_id' and 'study_version' in our `passive_data` data frame, we need to extract these data from the monitoring sheet. This is done in section 2

2. Additionally the `monitoring_data` data frame is set up in section 2


### 1.5 Infer Timezone Offset

In [ ]:
df_tz = create_utcday_tzoffset_df(df_backup_recent)


In [ ]:
df_tz["inferred_tzoffset_timedelta"] = pd.to_timedelta(
    df_tz["inferred_tzoffset"], unit="min"
)

In [ ]:
df_tz.columns

In [ ]:
df_backup_recent = df_backup_recent.merge(
    df_tz,
    left_on=["id", "start_date"],
    right_on=["id", "day"],
    how="left",
)
#df_backup_recent.drop(columns=["day"], inplace=True)  # remove day from df_tz

In [ ]:
df_backup_recent["local_timestamp_start"] = (
    df_backup_recent["timestamp_start"] + df_backup_recent["inferred_tzoffset_timedelta"]
).dt.tz_localize(None)

df_backup_recent["local_timestamp_end"] = (
    df_backup_recent["timestamp_end"] + df_backup_recent["inferred_tzoffset_timedelta"]
).dt.tz_localize(None)

df_backup_recent.head()

In [ ]:
assert df_backup_recent.inferred_tzoffset.isna().sum() == 0, (
    "There are missing inferred timezone offsets!"
)

In [ ]:
# just to make sure that we don't use them anymore later
del df_complete_filtered
del df_complete

## 2. Monitoring data

In [ ]:
# import data
df_monitoring = pd.read_csv(f"https://docs.google.com/spreadsheets/d/{proj_sheet}/export?format=csv")

In [ ]:
# get an overview of the monitoring data
#df_monitoring.head()

In [ ]:
# show columns of monitoring data
print(df_monitoring.columns.tolist())

In [ ]:
df_monitoring = df_monitoring.copy()

df_monitoring.rename(columns = {"Pseudonym": "id",
                                "FOR_ID": "for_id",
                                "EMA_ID": "ema_id", 
                                "Status": "study_status",
                                "Studienversion":"study_version", 
                                "Start EMA Baseline": "ema_base_start", 
                                "Ende EMA Baseline": "ema_base_end", 
                                "Freischaltung/ Start EMA T20": "ema_t20_start",
                                "Ende EMA T20":"ema_t20_end", 
                                "Freischaltung/ Start EMA Post":"ema_post_start",
                               "Ende EMA Post":"ema_post_end", 
                                "T20=Post":"t20_post" }, 
                     inplace=True)

df_monitoring = df_monitoring[['for_id', 'ema_id', 'id', 'study_version', 'study_status',
       't20_post', 'ema_base_start', 'ema_base_end', 'ema_t20_start', 'ema_t20_end',
       'ema_post_start', 'ema_post_end']]

df_monitoring["id"] = df_monitoring["id"].str[:4]
df_monitoring["for_id"] = df_monitoring.for_id.str.strip()

df_monitoring["ema_base_start"] = pd.to_datetime(
    df_monitoring["ema_base_start"], dayfirst=True, errors="coerce", utc=True
)
df_monitoring["ema_base_end"] = pd.to_datetime(
    df_monitoring["ema_base_end"], dayfirst=True, errors="coerce", utc=True
)


### 2.1 Merge relevant columns with passive data

In [ ]:
df_monitoring_short = df_monitoring[["id", "for_id","study_version"]]

#### 2.2 Final `passive_data` data frame

In [ ]:
df_backup_recent= df_backup_recent.merge(df_monitoring_short, on="id", how="right")

In [ ]:
df_backup_recent.head()

In [ ]:
# ensure data types are coded correctly
df_backup_recent['boolean_value'] = df_backup_recent['boolean_value'].astype('boolean[pyarrow]')
df_backup_recent['study_version'] = df_backup_recent['study_version'].astype('string')
df_backup_recent['modality'] = df_backup_recent['modality'].astype('string')
df_backup_recent['id'] = df_backup_recent['id'].astype('category')

In [ ]:
# Get a list of columns to drop (all columns not in keep_cols)
keep_cols_passive = ['id','for_id', 'modality', 'timestamp_start','timestamp_end',
    'local_timestamp_start', 'local_timestamp_end','time_interval', 'float_value', 'boolean_value','start_date', 
    'start_hour', "timezone_offset", 'study_version']

# final passive data frame
df_passive_final = df_backup_recent[keep_cols_passive]

#### 2.3 Final `monitoring_data` data frame

In [ ]:
#??

## 3. EMA data

#### 3.1 Load, match and rename relevant data from separate .csv files

In [ ]:

# Beispiel: datapath = Path("/pfad/zum/verzeichnis")
session        = pd.read_csv(datapath / "questionnaireSession.csv", low_memory=False)
answers        = pd.read_csv(datapath / "answers.csv", low_memory=False)
choice         = pd.read_csv(datapath / "choice.csv", low_memory=False)
questions      = pd.read_csv(datapath / "questions.csv", low_memory=False)
questionnaire  = pd.read_csv(datapath / "questionnaires.csv", low_memory=False)  # ohne Komma!


In [ ]:
# session data
study_mapping = {24: 0, 25: 0, 33: 1, 34: 1, 38: 2, 39: 2}
chronotype_mapping = {24: 0, 25: 1, 33: 0, 34: 1, 38: 0, 39: 1} 
session["user"] = session["user"].str[:4]
session.rename(columns = {"id":"session_id",
                          "user":"id",
                          "completedAt": "timestamp_beep_completion", 
                          "createdAt": "timestamp_item_completion", 
                          "expirationTimestamp": "timestamp_beep_expiration",
                          "sessionRun":"beep_num_run",
                          "study":"schedule_chronotype"}, inplace=True)
session['measurement_burst'] = session['schedule_chronotype'].map(study_mapping)
session['schedule_chronotype'] = session['schedule_chronotype'].map(chronotype_mapping)
# Parse epoch‑ms columns as UTC and drop tz info -> naive UTC
for col in ["timestamp_item_completion", "timestamp_beep_completion", "timestamp_beep_expiration"]:
    session[col] = (
        pd.to_datetime(session[col], unit="ms", utc=True, errors="coerce")
    )

df_sess = session[["id","session_id", "measurement_burst", "beep_num_run", "timestamp_item_completion", "timestamp_beep_completion", "schedule_chronotype", "timestamp_beep_expiration"]]

In [ ]:
study_mapping = {24: 0, 25: 0, 33: 1, 34: 1, 38: 2, 39: 2}
chronotype_mapping = {24: 0, 25: 1, 33: 0, 34: 1, 38: 0, 39: 1} 

answers["user"] = answers["user"].str[:4]
answers = answers[["user", "questionnaire", "study", "question", "element",
                   "createdAt", "order", "questionnaireSession"]]

answers["createdAt"] = (
    pd.to_datetime(answers["createdAt"], unit="ms", utc=True, errors="coerce")
)
answers['measurement_burst'] = answers['study'].map(study_mapping)
answers['schedule_chronotype'] = answers['study'].map(chronotype_mapping)

answers.rename(columns={"user": "id", 
                        "createdAt": "timestamp_item_completion",
                        "questionnaire": "beep_type",
                        "question":"item_code_map",
                        "order":"item_order",
                        "questionnaireSession":"session_id",
                        }, inplace=True)
answers.drop(columns=["study"], inplace=True)

In [ ]:
# item description data
choice = choice[["element", "choice_id", "text", "question"]]
choice.rename(columns={"text":"response_text",
                       "choice_id":"response", 
                       "question":"item_code_map"}, inplace=True)

In [ ]:
# question description data
questions = questions[["id", "title"]]
questions.rename(columns={"id":"item_code_map",
                          "title":"item"}, inplace=True)

In [ ]:
questionnaire = questionnaire[["id", "name"]]
questionnaire.rename(columns={"id":"beep_type",
                              "name":"beep_type_name"}, inplace=True)

In [ ]:
answer_merged = pd.merge(answers, choice, on= ["item_code_map","element"])
answer_merged = pd.merge(answer_merged, questions, on= "item_code_map")
answer_merged = pd.merge(answer_merged, questionnaire, on= "beep_type")
answer_merged["date"] = answer_merged["timestamp_item_completion"].dt.normalize()

#### 3.2 Calculate auxiliary variables

In [ ]:
df_monitoring_ema = df_monitoring[["id", "for_id","study_version", 'ema_base_start', 'ema_base_end',
       'ema_t20_start', 'ema_t20_end', 'ema_post_start', 'ema_post_end', 't20_post']]

In [ ]:
bursts = [("base", 0), ("t20", 1), ("post", 2)]

out = []
for burst_name, burst_code in bursts:
    tmp = df_monitoring_ema[[
        "id", "for_id", "study_version", "t20_post",
        f"ema_{burst_name}_start", f"ema_{burst_name}_end"
    ]].copy()

    tmp = tmp.rename(columns={
        f"ema_{burst_name}_start": "ema_burst_start",
        f"ema_{burst_name}_end":   "ema_burst_end",
    })
    tmp["measurement_burst"] = burst_code
    out.append(tmp)

df_monitoring_ema_long = (
    pd.concat(out, ignore_index=True)
      # optional: drop rows where the burst is entirely missing
      .dropna(subset=["ema_burst_start", "ema_burst_end"], how="all")
      .sort_values(["id", "measurement_burst"])
      .reset_index(drop=True)
)


In [ ]:
answer_merged = pd.merge(answer_merged, df_monitoring_ema_long, on = ["id", "measurement_burst"])

In [ ]:
df_ema_content = answer_merged.copy()

In [ ]:
meta_cols = ['id','for_id','date','response_text','item_code_map','beep_type' ,'beep_type_name',
              'element', 'item_order', 'session_id']
df_ema_meta = df_ema_content[meta_cols].copy()

In [ ]:
df_ema_content = df_ema_content.drop(columns=['response_text','item_code_map','beep_type' ,'beep_type_name',
              'element', 'item_order']) 

In [ ]:
df_sess_short = df_sess[["id", "session_id",  "beep_num_run",'timestamp_beep_completion']].copy()

In [ ]:
df_ema_meta = df_ema_meta.merge(df_sess_short, on=["id", "session_id"], how="left")

#### 3.3 Calculate EMA coverage

In [ ]:
df_sess_short_compliance = df_sess[["id", "session_id", 'timestamp_beep_completion']].copy()

In [ ]:
df_ema_content = df_ema_content.merge(df_sess_short_compliance, on=["id", "session_id"], how="left")

In [ ]:
df_ema_content["n_beeps_beeps_completed_per_burst"] = (
    df_ema_content
    .groupby(["measurement_burst", "id"])["timestamp_beep_completion"]
    .transform("nunique"))

#### 3.3 Calculate auxiliary variables

In [ ]:
df_ema_content = answer_merged.copy()

In [ ]:
# 1. Date and Time Manipulations
df_ema_content['weekday'] = df_ema_content['timestamp_item_completion'].dt.day_name()


# 1a. Season
def get_season(month):
    if month in [3, 4, 5]:
        return 1
    elif month in [6, 7, 8]:
        return 2
    elif month in [9, 10, 11]:
        return 3
    else:
        return 4

df_ema_content['season'] = df_ema_content['timestamp_item_completion'].dt.month.apply(get_season)

# 1b. Time of Day
def get_time_of_day(hour):
    if 5 <= hour < 8:
        return 1
    elif 8 <= hour < 12:
        return 2
    elif 12 <= hour < 17:
        return 3
    elif 17 <= hour < 21:
        return 4
    else:
        return 5

df_ema_content['time_of_day'] = df_ema_content['timestamp_item_completion'].dt.hour.apply(get_time_of_day)
df_ema_content['item'] = df_ema_content['item'].str.replace('_morning', '', regex=False)

# 3. Weekend Indicator
df_ema_content['weekend'] = df_ema_content['weekday'].isin(['Saturday', 'Sunday']).astype(int)

# 4. Questionnaire Number
df_ema_content['nr_beep_daily'] = df_ema_content['beep_type_name'].str.extract(r'(\d+)').astype(float)

# 5. Count unique questionnaires per day
df_ema_content['n_beeps_completed_per_day'] = (
    df_ema_content.groupby(['measurement_burst', 'id', 'date'])['beep_type_name']
                  .transform('nunique')
)

# 6. Unique Day Identifier
df_ema_content['quest_nr_str'] = df_ema_content['nr_beep_daily'].fillna('unknown').astype(str)
df_ema_content['beep_per_person_id'] = (
    df_ema_content['date'].dt.strftime('%Y%m%d') + '_' + df_ema_content['quest_nr_str']
)

In [ ]:
# 7. (ersetzt) Relative Start/End pro Phase & Customer
df_ema_content['ema_relative_start'] = (
    df_ema_content.groupby(['id', 'measurement_burst'])['date'].transform('min')
)
df_ema_content['ema_relative_end'] = (
    df_ema_content.groupby(['id', 'measurement_burst'])['date'].transform('max')
)

# 8. Absolute & Relative Day Index
df_ema_content['absolute_day_index'] = (
    df_ema_content['date'] - df_ema_content['ema_relative_start']
).dt.days + 1

df_ema_content['relative_day_index'] = (
    df_ema_content.groupby(['id', 'measurement_burst'])['date']
                  .rank(method='dense').astype(int)
)


In [ ]:

# 9. Filter absolute_day_index > 16
max_allowed_days = 16
#df_ema_content = df_ema_content[df_ema_content['absolute_day_index'] <= max_allowed_days].reset_index(drop=True)

# 10. Check
high_indices = df_ema_content[df_ema_content['absolute_day_index'] > max_allowed_days]
high_indices_id = high_indices.id.unique().tolist()
if not high_indices.empty:
    print("Warning: High absolute_day_index vorhanden:", high_indices['id'].unique())
else:
    print("All entries have absolute_day_index <= 16.")

In [ ]:
df_max_day = (
    df_ema_content.loc[
        df_ema_content["id"].isin(high_indices_id) & (df_ema_content["absolute_day_index"] > 16),
        ["id", "measurement_burst", "ema_relative_start", "ema_relative_end", "absolute_day_index", "beep_per_person_id"]
    ]
    .groupby(["id", "measurement_burst", "ema_relative_start", "ema_relative_end"], as_index=False)
    .agg(
        max_absolute_day_index=("absolute_day_index", "max"),
        n_unique_beeps_after_day16=("beep_per_person_id", "nunique"),
    )
    .sort_values(["id", "measurement_burst", "max_absolute_day_index"])
)

df_max_day_burst0 = df_max_day.loc[df_max_day["measurement_burst"] == 0]
df_max_day_burst1 = df_max_day.loc[df_max_day["measurement_burst"] == 1]
df_max_day_burst2 = df_max_day.loc[df_max_day["measurement_burst"] == 2]

df_max_day_burst0


In [ ]:
df_max_day_burst1

In [ ]:
df_max_day_burst2

In [ ]:
# Remove Beeps after Day 16 for the Measurement Burst 0
df_ema_content = df_ema_content.loc[
    ~(
        (df_ema_content["measurement_burst"] == 0) &
        (df_ema_content["absolute_day_index"] > 16)
    )
].copy()

In [ ]:
# Remove out of phase data for Measurement Burst 1 and 2 (self-activation was possible, leading to false-activation of study phases)


In [ ]:
# 11. Questionnaire Counter
df_unique = df_ema_content.drop_duplicates(subset=['id', 'measurement_burst', 'beep_per_person_id']).copy()
df_unique['relative_beep_counter'] = df_unique.groupby(['id', 'measurement_burst']).cumcount() + 1
df_ema_content = df_ema_content.merge(
    df_unique[['id', 'measurement_burst', 'beep_per_person_id', 'relative_beep_counter']],
    on=['id', 'measurement_burst', 'beep_per_person_id'],
    how='left'
)

# 12. Missing Data behandeln
df_ema_content['measurement_burst'] = df_ema_content['measurement_burst'].fillna('unknown')
df_ema_content['absolute_day_index'] = df_ema_content['absolute_day_index'].where(
    df_ema_content['ema_relative_start'].notna(), np.nan
)

### 3.4 merge the inferred tz offsets

In [ ]:
# uncomment if want to run this cell multiple times
# if "infered_tzoffset" in df_sess.columns:
#     print("Dropping existing 'inferred_tzoffset' columns from df_sess")
#     df_sess = df_sess.drop(columns=["inferred_tzoffset", "inferred_tzoffset_source"])
# if "inferred_tzoffset" in df_ema_content.columns:
#     print("Dropping existing 'inferred_tzoffset' columns from df_ema_content")
#     df_ema_content = df_ema_content.drop(columns=["inferred_tzoffset", "inferred_tzoffset_source"])

df_ema_meta = merge_fill_tz(
    df_ema_meta, df_tz, day_col="date", customer_col="id"
)
df_ema_content = merge_fill_tz(
    df_ema_content, df_tz, day_col="date", customer_col="id"
)

In [ ]:
df_ema_content.drop(columns=['element', 'item_order', 'session_id'], inplace=True) 

### Export passive, EMA and Monitoring

In [ ]:
backup_path = raw_path + "/backup_passive_recent.feather"
df_passive_final.to_feather(backup_path)

preprocessed_path_final = preprocessed_path + "/backup_passive_recent.feather"
df_passive_final.to_feather(preprocessed_path_final)

#preprocessed_path_freezed_final = preprocessed_path_freezed + "/code_check" + "/backup_passive_recent.parquet"
#df_passive_final.to_parquet(preprocessed_path_freezed_final)

with open(preprocessed_path + '/ema_meta.pkl', 'wb') as file:
    pickle.dump(df_ema_meta, file)

    
with open(preprocessed_path + '/monitoring_data.pkl', 'wb') as file:
    pickle.dump(df_monitoring, file)

    
with open(preprocessed_path + '/ema_content.pkl', 'wb') as file:
    pickle.dump(df_ema_content, file)

In [ ]:

# Export ema meta as CSV
df_ema_path = preprocessed_path + '/ema_meta.csv'
df_ema_meta.to_csv(df_ema_path, index=False)

# Export df_monitoring as CSV
df_monitoring_csv_path = preprocessed_path + '/monitoring_data.csv'
df_monitoring.to_csv(df_monitoring_csv_path, index=False)

# Export df_ema_content as CSV
df_ema_content_csv_path = preprocessed_path + '/ema_content.csv'
df_ema_content.to_csv(df_ema_content_csv_path, index=False)

# Export df_ema_content as CSV to freezed for data check
#df_ema_content_csv_path = preprocessed_path_freezed +'/code_check' +'/ema_content_recent.csv'
#df_ema_content.to_csv(df_ema_content_csv_path, index=False)

